# Logistic Regression for Diabetes Prediction

This notebook solves the Week 8 Day 4 ExerciseXP using a logistic regression model to predict diabetes.

The dataset is downloaded automatically from the provided ZIP file URL, then explored, split, standardized, trained, and evaluated.

## Exercise 1 - Understanding the problem and data collection

In [ ]:
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

DATA_URL = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%204/Day%202/Diabetes%20prediction%20dataset.zip"
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
zip_path = data_dir / "diabetes_prediction_dataset.zip"
extract_dir = data_dir / "diabetes_prediction_dataset"

if not zip_path.exists():
    urlretrieve(DATA_URL, zip_path)

extract_dir.mkdir(parents=True, exist_ok=True)

csv_files = list(extract_dir.rglob("*.csv"))
if not csv_files:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    csv_files = list(extract_dir.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV file found in the downloaded archive.")

csv_path = csv_files[0]
df = pd.read_csv(csv_path)

df.head()

In [ ]:
df.info()

display(df.describe())

target_col = "Outcome" if "Outcome" in df.columns else df.columns[-1]
positive_cases = int(df[target_col].sum())
negative_cases = int(len(df) - positive_cases)

print(f"Target column: {target_col}")
print(f"Positive cases: {positive_cases}")
print(f"Negative cases: {negative_cases}")

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Exercise 2 - Model picking and standardization

A logistic regression model is a good choice here because the task is a binary classification problem.

Yes, the data should be standardized because logistic regression is sensitive to feature scales.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Exercise 3 - Model training

In [ ]:
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train_scaled, y_train)

y_pred = log_reg_model.predict(X_test_scaled)
y_proba = log_reg_model.predict_proba(X_test_scaled)[:, 1]

## Exercise 4 - Evaluation metrics

The score, confusion matrix, and classification report help us understand how well the model separates positive and negative cases.

In [ ]:
train_accuracy = accuracy_score(y_train, log_reg_model.predict(X_train_scaled))
test_accuracy = accuracy_score(y_test, y_pred)

accuracy_df = pd.DataFrame({
    "Split": ["Train", "Test"],
    "Accuracy": [train_accuracy, test_accuracy],
})

ax = accuracy_df.plot(
    kind="bar",
    x="Split",
    y="Accuracy",
    legend=False,
    color=["#2E86AB", "#F18F01"],
    figsize=(6, 4),
)
ax.set_ylim(0, 1)
ax.set_title("Accuracy Score")
ax.set_ylabel("Accuracy")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f")
plt.tight_layout()
plt.show()

print(f"Train accuracy: {train_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["No diabetes", "Diabetes"],
    yticklabels=["No diabetes", "Diabetes"],
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred))

## Exercise 5 - Visualizing the performance of the model

The decision boundary below is built on the first two PCA components so it can be visualized in 2D. This is for illustration only; the main model still uses all standardized features.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

boundary_model = LogisticRegression(max_iter=1000, random_state=42)
boundary_model.fit(X_train_pca, y_train)

pca_train_accuracy = accuracy_score(y_train, boundary_model.predict(X_train_pca))
pca_test_accuracy = accuracy_score(y_test, boundary_model.predict(X_test_pca))

x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
z = boundary_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, z, alpha=0.25, cmap="coolwarm")
scatter = plt.scatter(
    X_test_pca[:, 0],
    X_test_pca[:, 1],
    c=y_test,
    cmap="coolwarm",
    edgecolor="k",
    s=35,
)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title(f"Decision Boundary (Test accuracy: {pca_test_accuracy:.3f})")
plt.legend(*scatter.legend_elements(), title="Diabetes")
plt.tight_layout()
plt.show()

print(f"PCA train accuracy: {pca_train_accuracy:.3f}")
print(f"PCA test accuracy: {pca_test_accuracy:.3f}")

## Exercise 6 - ROC curve

The ROC curve shows the trade-off between the true positive rate and false positive rate at different thresholds.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="#2E86AB", label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"ROC AUC: {roc_auc:.3f}")

## Conclusion

The logistic regression model provides a simple and interpretable baseline for diabetes prediction. The evaluation plots and metrics help judge how well it performs on unseen data.